In [ ]:
# ============================================================
# Statistical significance tests
# ============================================================

!pip install -q pandas numpy scipy

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 1. Load full results file
# ============================================================

FULL_RESULTS_PATH = "results/full_results.csv"

OUT_DIR = Path("results/statistical_tests")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(FULL_RESULTS_PATH)

print("Rows:", len(df))
print("Methods:", df["method"].nunique())
print(df["method"].unique())

In [ ]:
# ============================================================
# 2. Clean method labels
# ============================================================

METHOD_LABELS = {
    "mazare2018_retrieval_only_raw_700k": "Semantic Persona Retrieval (Mazaré et al.)",
    "partner_matters_persona_fusion_response_selection": "Persona Fusion Retrieval (Partner Matters)",
    "coo_style_usefulness_ranking_raw_700k": "Usefulness-Aware Persona Selection (COO)",
    "personadb_offline_hierarchical_refinement": "Hierarchical Persona Refinement (Persona-DB)",
    "ours_stance_nli_gpt_final_repository": "Contradiction-Aware Persona Retrieval (Ours)",
}

OURS_METHOD = "ours_stance_nli_gpt_final_repository"

df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])

In [ ]:
# ============================================================
# 3. Create stable paired case id
# ============================================================

# Each method was evaluated on the same test cases.
# We identify each case by topic + user opinion + expected stance.

df["case_id"] = (
    df["topic"].astype(str)
    + " || "
    + df["expected_user_stance"].astype(str)
    + " || "
    + df["user_opinion"].astype(str)
)

print("Unique test cases:", df["case_id"].nunique())

In [ ]:
# ============================================================
# Paired significance test function
# ============================================================

def paired_significance_tests(
    df,
    metric_col,
    ours_method=OURS_METHOD,
    alternative="greater",
):
    """
    Runs paired Wilcoxon signed-rank tests comparing ours vs every baseline.

    alternative="greater" tests whether ours > baseline.
    Use alternative="two-sided" for a more conservative non-directional test.
    """

    pivot = df.pivot_table(
        index="case_id",
        columns="method",
        values=metric_col,
        aggfunc="mean",
    )

    if ours_method not in pivot.columns:
        raise ValueError(f"Could not find ours method: {ours_method}")

    rows = []
    ours_values = pivot[ours_method]

    for baseline in pivot.columns:
        if baseline == ours_method:
            continue

        pair_df = pd.DataFrame({
            "ours": ours_values,
            "baseline": pivot[baseline],
        }).dropna()

        x = pair_df["ours"].to_numpy(dtype=float)
        y = pair_df["baseline"].to_numpy(dtype=float)
        diff = x - y

        if len(pair_df) == 0:
            stat = np.nan
            p_value = np.nan
        elif np.allclose(diff, 0):
            stat = np.nan
            p_value = 1.0
        else:
            stat, p_value = wilcoxon(
                x,
                y,
                alternative=alternative,
                zero_method="wilcox",
            )

        rows.append({
            "metric": metric_col,
            "comparison": f"{METHOD_LABELS.get(ours_method, ours_method)} vs {METHOD_LABELS.get(baseline, baseline)}",
            "ours_method": ours_method,
            "baseline_method": baseline,
            "ours_label": METHOD_LABELS.get(ours_method, ours_method),
            "baseline_label": METHOD_LABELS.get(baseline, baseline),
            "n_pairs": len(pair_df),
            "ours_mean": float(np.mean(x)) if len(x) else np.nan,
            "baseline_mean": float(np.mean(y)) if len(y) else np.nan,
            "mean_difference": float(np.mean(diff)) if len(diff) else np.nan,
            "wilcoxon_statistic": float(stat) if not pd.isna(stat) else np.nan,
            "p_value": float(p_value) if not pd.isna(p_value) else np.nan,
            "significant_p_0_05": bool(p_value < 0.05) if not pd.isna(p_value) else False,
        })

    return pd.DataFrame(rows).sort_values("p_value")


In [ ]:
# ============================================================
# 6. Run tests for main metrics
# ============================================================

nli_tests_df = paired_significance_tests(
    df=df,
    metric_col="nli_contradiction",
    alternative="greater",
)

stance_tests_df = paired_significance_tests(
    df=df,
    metric_col="opposite_stance_accuracy",
    alternative="greater",
)

display(nli_tests_df)
display(stance_tests_df)

In [ ]:
# ============================================================
# 7. Optional: two-sided tests
# ============================================================

nli_tests_two_sided_df = paired_significance_tests(
    df=df,
    metric_col="nli_contradiction",
    alternative="two-sided",
)

stance_tests_two_sided_df = paired_significance_tests(
    df=df,
    metric_col="opposite_stance_accuracy",
    alternative="two-sided",
)

display(nli_tests_two_sided_df)
display(stance_tests_two_sided_df)

In [ ]:
# ============================================================
# 8. Save results
# ============================================================

nli_tests_df.to_csv(
    OUT_DIR / "wilcoxon_nli_contradiction_greater.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_df.to_csv(
    OUT_DIR / "wilcoxon_opposite_stance_accuracy_greater.csv",
    index=False,
    encoding="utf-8-sig",
)

nli_tests_two_sided_df.to_csv(
    OUT_DIR / "wilcoxon_nli_contradiction_two_sided.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_two_sided_df.to_csv(
    OUT_DIR / "wilcoxon_opposite_stance_accuracy_two_sided.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved statistical test results to:", OUT_DIR)

In [ ]:
# ============================================================
# Create publication-friendly tables
# ============================================================

KEEP_COLS = [
    "comparison",
    "n_pairs",
    "ours_mean",
    "baseline_mean",
    "mean_difference",
    "p_value",
    "significant_p_0_05",
]

nli_tests_paper_df = nli_tests_df[KEEP_COLS].copy()
stance_tests_paper_df = stance_tests_df[KEEP_COLS].copy()

display(nli_tests_paper_df)
display(stance_tests_paper_df)

In [ ]:
nli_tests_paper_df.to_csv(
    OUT_DIR / "nli_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_paper_df.to_csv(
    OUT_DIR / "stance_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved paper-friendly significance tables.")

In [ ]:
nli_tests_paper_df = nli_tests_df[
    [
        "comparison",
        "mean_difference",
        "p_value",
        "significant_p_0_05",
    ]
].copy()

stance_tests_paper_df = stance_tests_df[
    [
        "comparison",
        "mean_difference",
        "p_value",
        "significant_p_0_05",
    ]
].copy()

In [ ]:
nli_tests_paper_df.to_csv(
    OUT_DIR / "summary_nli_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

stance_tests_paper_df.to_csv(
    OUT_DIR / "summary_stance_significance_table_for_paper.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved paper-friendly significance tables.")